# 09 — Day 9 Feature Extraction: Pitch-Shifted WavLM (Layer 6)

**Standalone**, same pattern as `08a_features_wavlm_augmented.ipynb`. Tests a different acoustic
axis than Day 8a: **pitch**, not tempo. Day 8a's speed perturbation changed pitch and tempo
*together* (a side effect of resampling) and found augmentation doesn't help WavLM the way it
helped HuBERT — plausibly because WavLM's fatigue signal leans on speaking rate, and forcing
tempo-invariance cost more than the added training volume gained. Pitch-shifting changes *only*
pitch, leaving tempo/duration untouched, so this is a genuinely different bet: if WavLM's signal is
tempo-dependent but not pitch-dependent, pitch-shift augmentation could still help even though
speed-shift didn't.

**What this does:** for each of the 280 recordings, create two pitch-shifted copies — **2 semitones
up** and **2 semitones down** — via `librosa.effects.pitch_shift` (phase-vocoder based; preserves
duration exactly, unlike the resample-based speed perturbation Day 5/8a used). Each copy runs
through WavLM-base-plus **layer 6 only** (Day 7's winner). The unperturbed (original) embeddings are
reused from Day 7's cache, not re-extracted, same as every prior augmentation-extraction notebook.

**±2 semitones, stated as a deliberate choice:** roughly comparable in perceptual magnitude to Day
5/8a's ±10% speed change — noticeable but not distorting. Not tuned; a reasonable first choice to
test the isolated pitch axis, not a swept hyperparameter.

**Inputs needed:** just the `Fatigue_data` dataset — no need to attach Day 7's output here.

## 0. Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import librosa

DATA_DIR = next(Path("/kaggle/input").rglob("project_data"))
AUDIO_DIR = DATA_DIR / "audio"
LABELS_CSV = DATA_DIR / "labels.csv"

OUT_DIR = Path("/kaggle/working/features")
OUT_DIR.mkdir(parents=True, exist_ok=True)

SAMPLE_RATE = 16000
LAYER = 6
PITCH_SHIFTS = [-2, 2]  # semitones; 0 (original) reused from Day 7's cache, not regenerated

labels = pd.read_csv(LABELS_CSV)
missing = [f for f in labels["filename"] if not (AUDIO_DIR / f).exists()]
assert not missing, f"{len(missing)} filenames missing audio files: {missing[:5]}"
assert len(labels) == 280, f"expected 280 rows, got {len(labels)}"

print(f"DATA_DIR  = {DATA_DIR}")
print(f"AUDIO_DIR = {AUDIO_DIR}")
print(f"Loaded {len(labels)} labeled recordings.")

## 1. Load WavLM (layer 6 only)

In [ ]:
!pip install -q transformers

In [ ]:
import torch
from transformers import Wav2Vec2FeatureExtractor, WavLMModel

WAVLM_MODEL_NAME = "microsoft/wavlm-base-plus"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(WAVLM_MODEL_NAME)
model = WavLMModel.from_pretrained(WAVLM_MODEL_NAME).to(device)
model.eval()  # frozen, same as every other extraction notebook in this project

## 2. Pitch Shift & Extraction

`librosa.effects.pitch_shift` shifts pitch by a number of semitones while preserving duration —
mechanically different from Day 5/8a's resample-based speed perturbation, which changed pitch and
tempo together. This isolates pitch as its own variable.

In [ ]:
def pitch_shift(y: np.ndarray, sr: int, n_steps: int) -> np.ndarray:
    """Duration-preserving pitch shift (phase vocoder). n_steps in semitones;
    positive = higher pitch, negative = lower pitch. Tempo is untouched."""
    return librosa.effects.pitch_shift(y, sr=sr, n_steps=n_steps)


def extract_wavlm_layer6(y: np.ndarray) -> np.ndarray:
    """Frozen WavLM forward pass -> layer 6 hidden states -> mean+std pool.
    Identical to 08a_features_wavlm_augmented.ipynb's extraction function."""
    inputs = feature_extractor(y, sampling_rate=SAMPLE_RATE, return_tensors="pt")
    input_values = inputs.input_values.to(device)

    with torch.no_grad():
        outputs = model(input_values, output_hidden_states=True)
        hidden = outputs.hidden_states[LAYER].squeeze(0)  # (T, 768)

    mean_pool = hidden.mean(dim=0)
    std_pool = hidden.std(dim=0)
    return torch.cat([mean_pool, std_pool]).cpu().numpy()  # (1536,)

In [ ]:
from tqdm.auto import tqdm

rows = []
for filename in tqdm(labels["filename"], desc="Extracting pitch-shifted embeddings"):
    y, _ = librosa.load(AUDIO_DIR / filename, sr=SAMPLE_RATE, mono=True)
    y_trimmed, _ = librosa.effects.trim(y, top_db=25)  # trim BEFORE shifting, same as every other notebook

    for n_steps in PITCH_SHIFTS:
        y_shifted = pitch_shift(y_trimmed, SAMPLE_RATE, n_steps)
        emb = extract_wavlm_layer6(y_shifted)
        rows.append({
            "source_filename": filename,
            "pitch_shift": n_steps,
            **{f"emb_{i}": v for i, v in enumerate(emb)},
        })

augmented_df = pd.DataFrame(rows)
print(augmented_df.shape)  # expect (560, 2 + 1536)
augmented_df.head()

## 3. Cache & Verify

In [ ]:
out_path = OUT_DIR / "wavlm_layer6_pitchshift_augmented.parquet"
augmented_df.to_parquet(out_path, index=False)

assert len(augmented_df) == 280 * len(PITCH_SHIFTS), \
    f"expected {280 * len(PITCH_SHIFTS)} rows, got {len(augmented_df)}"
n_emb_cols = sum(c.startswith("emb_") for c in augmented_df.columns)
assert n_emb_cols == 1536, f"expected 1536 embedding dims, got {n_emb_cols}"
assert set(augmented_df["source_filename"]) == set(labels["filename"]), "filename mismatch vs labels.csv"
assert set(augmented_df["pitch_shift"].unique()) == set(PITCH_SHIFTS)

print(f"Saved {out_path}")
print(f"Shape: {augmented_df.shape} (280 recordings x {len(PITCH_SHIFTS)} pitch variants x 1536 dims)")
print("All verification checks passed.")

## Next steps

1. **Save this notebook's version.**
2. Open `09_evaluate_wavlm_pitchshift.ipynb`, attach **all three**: this notebook's output
   (`wavlm_layer6_pitchshift_augmented.parquet`), **Day 7's `07_features_wavlm.ipynb`** output (for
   the original/unperturbed layer-6 embeddings), and the `Fatigue_data` dataset. It tests
   pitch-shift augmentation isolated, then combined with session-fusion, against Day 7's plain
   0.663 and Day 8a's speed-augmented results (0.643 baseline / 0.638 fusion).